# AutoGaze × Pretrained CV Decoders

AutoGaze가 선택한 토큰만 사용했을 때 **기존 학습된 decoder**의 성능이 어떻게 달라지는지 확인합니다.

**핵심 아이디어**
- AutoGaze는 14×14=196개 공간 위치에 대한 gaze score(중요도)를 출력합니다.
- 이 score를 target ViT의 patch grid 크기에 bilinear interpolation으로 맞춘 뒤,
  **forward hook**으로 선택되지 않은 patch token을 0으로 만듭니다.
- Pretrained decoder(DPT, YOLOS, DINOv2, SegFormer)는 **코드 수정 없이** 그대로 사용합니다.

```
입력 이미지
    ├─► AutoGaze (224×224) ──► 14×14 gaze map
    │                              │ bilinear interpolation
    │                              ▼
    └─► Pretrained ViT backbone ──► patch embeddings
                                        │ hook: zero-out non-selected tokens
                                        ▼
                               Pretrained Decoder (변경 없음)
                                        │
                                        ▼
                               Task output (depth / boxes / seg / class)
```

| 모델 | Task | 백본 | Patch grid |
|------|------|------|------------|
| Depth Anything V2 Small | Depth Estimation | DINOv2 ViT-S/14 | 37×37 @ 518px |
| YOLOS-Tiny | Object Detection | ViT-Tiny/16 | 32×32 @ 512px |
| DINOv2-Base | Recognition (lin. probe) | ViT-B/14 | 16×16 @ 224px |
| SegFormer-B2 | Semantic Seg | Mix-Transformer | 128×128 @ head |


In [ ]:
# ── 0. 공통 설정 ─────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import requests
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트
import matplotlib, platform
if platform.system() == 'Darwin':
    matplotlib.rc('font', family='AppleGothic')
matplotlib.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'device: {DEVICE}')

# ── AutoGaze 로드 ─────────────────────────────────────────────
from autogaze.models.autogaze import AutoGaze
from autogaze.models.autogaze.processing_autogaze import AutoGazeImageProcessor
from autogaze.models.autogaze.autogaze_cv import AutoGazeTokenSelector

AG_PATH = '../weights/AutoGaze'
ag_model = AutoGaze.from_pretrained(AG_PATH).to(DEVICE).eval()
ag_processor = AutoGazeImageProcessor.from_pretrained(AG_PATH)
selector = AutoGazeTokenSelector(ag_model, gazing_ratio=0.5)
print('AutoGaze loaded')

In [ ]:
# ── 1. 샘플 이미지 로드 ───────────────────────────────────────
# COCO val2017 sample (고양이 + 리모컨)
IMG_URL = 'http://images.cocodataset.org/val2017/000000039769.jpg'

try:
    resp = requests.get(IMG_URL, timeout=5)
    pil_img = Image.open(BytesIO(resp.content)).convert('RGB')
    print(f'이미지 로드 완료: {pil_img.size}')
except Exception:
    # 오프라인 fallback: 합성 이미지
    pil_img = Image.fromarray(np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8))
    print('오프라인: 합성 이미지 사용')

plt.figure(figsize=(6, 4))
plt.imshow(pil_img)
plt.axis('off')
plt.title('입력 이미지')
plt.show()

In [ ]:
# ── 2. AutoGaze Gaze Map 시각화 ───────────────────────────────
# AutoGaze용 전처리: 224×224 → (1, 1, C, 224, 224)
def prep_for_autogaze(pil_image, device):
    out = ag_processor(images=[pil_image.resize((224, 224))], return_tensors='pt')
    video = out['pixel_values'].unsqueeze(1).to(device)  # (1, 1, C, 224, 224)
    return video

ag_video = prep_for_autogaze(pil_img, DEVICE)

# Gaze map 시각화 함수
def show_gaze_map(ag_video, pil_image, title='AutoGaze gaze map'):
    with torch.no_grad():
        gaze_out = ag_model(
            {'video': ag_video}, gazing_ratio=0.5, generate_only=True
        )
    mask = torch.cat(gaze_out['gazing_mask'], dim=-1)[0, 0]  # (196,)
    gaze_map = mask.float().cpu().numpy().reshape(14, 14)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    axes[0].imshow(pil_image)
    axes[0].set_title('원본 이미지')
    axes[0].axis('off')

    im = axes[1].imshow(gaze_map, cmap='hot', vmin=0, vmax=1)
    plt.colorbar(im, ax=axes[1], fraction=0.046)
    axes[1].set_title('AutoGaze 14×14 gaze map\n(빨간색 = 선택된 토큰)')
    axes[1].set_xticks([]); axes[1].set_yticks([])

    # 원본 위에 gaze 오버레이
    img_arr = np.array(pil_image.resize((224, 224)))
    import cv2 as _cv
    overlay = _cv.resize(gaze_map, (224, 224), interpolation=_cv.INTER_LINEAR)
    axes[2].imshow(img_arr)
    axes[2].imshow(overlay, cmap='hot', alpha=0.5, vmin=0, vmax=1)
    axes[2].set_title('이미지 + gaze 오버레이')
    axes[2].axis('off')

    n_selected = int(mask.sum().item())
    fig.suptitle(f'{title}  |  선택 토큰: {n_selected}/196 ({n_selected/196*100:.0f}%)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    return gaze_map

try:
    import cv2
    gaze_map = show_gaze_map(ag_video, pil_img)
except ImportError:
    print('cv2 없음 — 오버레이 생략')
    with torch.no_grad():
        gaze_out = ag_model({'video': ag_video}, gazing_ratio=0.5, generate_only=True)
    mask = torch.cat(gaze_out['gazing_mask'], dim=-1)[0, 0]
    gaze_map = mask.float().cpu().numpy().reshape(14, 14)
    plt.imshow(gaze_map, cmap='hot'); plt.colorbar(); plt.title('Gaze map'); plt.show()

---
## Task 1: Depth Estimation — Depth Anything V2

- 모델: `depth-anything/Depth-Anything-V2-Small-hf`
- 백본: DINOv2 ViT-S, patch_size=14, 입력 518×518 → **37×37=1369 patch tokens**
- Hook 위치: `model.backbone.embeddings`
- AutoGaze 14×14 → **37×37** bilinear interpolation

In [ ]:
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

DA_MODEL = 'depth-anything/Depth-Anything-V2-Small-hf'
da_processor = AutoImageProcessor.from_pretrained(DA_MODEL)
da_model = AutoModelForDepthEstimation.from_pretrained(DA_MODEL).to(DEVICE).eval()
print(f'Depth Anything V2 Small loaded  |  patch_size=14')

# 입력 준비 (HF 전처리: 518×518)
da_inputs = da_processor(images=pil_img, return_tensors='pt')
da_inputs = {k: v.to(DEVICE) for k, v in da_inputs.items()}
inp_h = da_inputs['pixel_values'].shape[-2]
GRID_DA = inp_h // 14  # 518 // 14 = 37
print(f'입력 크기: {inp_h}×{inp_h}  →  patch grid: {GRID_DA}×{GRID_DA} = {GRID_DA**2} tokens')

In [ ]:
# Gaze mask 계산 (37×37)
mask_da = selector.compute_gaze_mask(ag_video, target_h=GRID_DA, target_w=GRID_DA)
print(f'Gaze mask: {mask_da.sum().item()}/{GRID_DA**2} tokens selected ({mask_da.float().mean()*100:.1f}%)')

ratios_to_test = [1.0, 0.75, 0.5, 0.25]
depth_results  = {}

# ratio=1.0: 원본 (마스크 없음)
with torch.no_grad():
    out_full = da_model(**da_inputs)
depth_results['전체 (100%)'] = out_full.predicted_depth.squeeze().cpu().float().numpy()

# ratio별로 테스트
for ratio in [0.75, 0.5, 0.25]:
    sel = AutoGazeTokenSelector(ag_model, gazing_ratio=ratio)
    m = sel.compute_gaze_mask(ag_video, target_h=GRID_DA, target_w=GRID_DA)
    with sel.token_mask_context(da_model.backbone.embeddings, m, has_cls_token=True):
        with torch.no_grad():
            out = da_model(**da_inputs)
    depth_results[f'AutoGaze {ratio*100:.0f}%'] = out.predicted_depth.squeeze().cpu().float().numpy()

# 시각화
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (label, depth) in zip(axes, depth_results.items()):
    im = ax.imshow(depth, cmap='Spectral_r')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.axis('off')

fig.suptitle('Depth Anything V2 — AutoGaze ratio별 depth map 비교',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 원본 대비 RMSE
ref = depth_results['전체 (100%)']
for label, depth in list(depth_results.items())[1:]:
    rmse = np.sqrt(((depth - ref) ** 2).mean())
    print(f'  {label}  →  RMSE vs full: {rmse:.4f}')

---
## Task 2: Object Detection — YOLOS

- 모델: `hustvl/yolos-tiny` (DETR 스타일, ViT-Tiny 백본)
- patch_size=16, 입력 512×512 → **32×32=1024 patch tokens**
- Hook 위치: `model.vit.embeddings`
- AutoGaze 14×14 → **32×32** bilinear interpolation

In [ ]:
from transformers import AutoImageProcessor, AutoModelForObjectDetection

YOLOS_MODEL = 'hustvl/yolos-tiny'
yolos_processor = AutoImageProcessor.from_pretrained(YOLOS_MODEL)
yolos_model = AutoModelForObjectDetection.from_pretrained(YOLOS_MODEL).to(DEVICE).eval()
print(f'YOLOS-Tiny loaded  |  patch_size=16')

yolos_inputs = yolos_processor(images=pil_img, return_tensors='pt')
yolos_inputs = {k: v.to(DEVICE) for k, v in yolos_inputs.items()}
inp_h_y = yolos_inputs['pixel_values'].shape[-2]
GRID_Y = inp_h_y // 16
print(f'입력 크기: {inp_h_y}px  →  patch grid: {GRID_Y}×{GRID_Y} = {GRID_Y**2} tokens')

In [ ]:
SCORE_THRESH = 0.3
COCO_COLORS  = plt.cm.tab20.colors

def draw_detections(ax, pil_image, logits, pred_boxes, processor, thresh=SCORE_THRESH):
    """Draw YOLOS detections on ax."""
    ax.imshow(pil_image)
    W, H = pil_image.size
    probs = logits.softmax(-1)[0, :, :-1]          # (num_det, num_cls)
    keep  = probs.max(-1).values > thresh
    boxes = pred_boxes[0, keep].cpu()
    labels = probs[keep].argmax(-1).cpu()
    scores = probs[keep].max(-1).values.cpu()

    for (cx, cy, w, h), lbl, sc in zip(boxes, labels, scores):
        x0 = (cx - w/2) * W;  y0 = (cy - h/2) * H
        bw = w * W;             bh = h * H
        color = COCO_COLORS[lbl % len(COCO_COLORS)]
        rect = mpatches.FancyBboxPatch(
            (x0, y0), bw, bh,
            boxstyle='round,pad=2', linewidth=2,
            edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        name = processor.id2label.get(lbl.item(), str(lbl.item()))
        ax.text(x0, y0 - 4, f'{name} {sc:.2f}',
                color='white', fontsize=8, fontweight='bold',
                bbox=dict(facecolor=color, alpha=0.8, pad=1))
    ax.axis('off')
    return int(keep.sum())


fig, axes = plt.subplots(1, 4, figsize=(20, 5))
n_boxes = {}

# 전체 토큰
with torch.no_grad():
    out_y = yolos_model(**yolos_inputs)
n = draw_detections(axes[0], pil_img, out_y.logits, out_y.pred_boxes, yolos_processor)
axes[0].set_title(f'전체 (100%)\n{n}개 박스', fontweight='bold')
n_boxes['100%'] = n

# AutoGaze ratio별
for ax, ratio in zip(axes[1:], [0.75, 0.5, 0.25]):
    sel = AutoGazeTokenSelector(ag_model, gazing_ratio=ratio)
    m = sel.compute_gaze_mask(ag_video, target_h=GRID_Y, target_w=GRID_Y)
    with sel.token_mask_context(yolos_model.vit.embeddings, m, has_cls_token=True):
        with torch.no_grad():
            out_y = yolos_model(**yolos_inputs)
    n = draw_detections(ax, pil_img, out_y.logits, out_y.pred_boxes, yolos_processor)
    ax.set_title(f'AutoGaze {ratio*100:.0f}%\n{n}개 박스', fontweight='bold')
    n_boxes[f'{ratio*100:.0f}%'] = n

fig.suptitle('YOLOS 객체 탐지 — AutoGaze ratio별 비교', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('박스 수:', n_boxes)

---
## Task 3: Recognition — DINOv2 Linear Probe

- 모델: `facebook/dinov2-base-imagenet1k-1-layer` (ImageNet-1K linear probe 포함)
- patch_size=14, 입력 224×224 → **16×16=256 patch tokens**
- Hook 위치: `model.dinov2.embeddings`
- AutoGaze 14×14 → **16×16** bilinear interpolation

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

DINO_MODEL = 'facebook/dinov2-base-imagenet1k-1-layer'
dino_processor = AutoImageProcessor.from_pretrained(DINO_MODEL)
dino_model = AutoModelForImageClassification.from_pretrained(DINO_MODEL).to(DEVICE).eval()
print(f'DINOv2-Base + ImageNet linear probe loaded  |  patch_size=14')

dino_inputs = dino_processor(images=pil_img, return_tensors='pt')
dino_inputs = {k: v.to(DEVICE) for k, v in dino_inputs.items()}
inp_h_d = dino_inputs['pixel_values'].shape[-2]
GRID_D = inp_h_d // 14
print(f'입력: {inp_h_d}×{inp_h_d}  →  patch grid: {GRID_D}×{GRID_D} = {GRID_D**2} tokens')

In [ ]:
def topk_classes(logits, processor, k=5):
    probs = logits.softmax(-1)[0]
    topk  = probs.topk(k)
    return [(processor.id2label[i.item()], p.item()) for i, p in zip(topk.indices, topk.values)]


print('─' * 60)
print(f'  {'레이블':35s}  {'전체':>6}  ', end='')

ratios_cls = [0.75, 0.5, 0.25]
for r in ratios_cls:
    print(f'  AG{r*100:.0f}%', end='')
print()
print('─' * 60)

# 전체 토큰 (baseline)
with torch.no_grad():
    out_base = dino_model(**dino_inputs)
base_top5 = topk_classes(out_base.logits, dino_processor, k=5)

# ratio별 확률 수집
ratio_top5 = {}
for r in ratios_cls:
    sel = AutoGazeTokenSelector(ag_model, gazing_ratio=r)
    m = sel.compute_gaze_mask(ag_video, target_h=GRID_D, target_w=GRID_D)
    with sel.token_mask_context(dino_model.dinov2.embeddings, m, has_cls_token=True):
        with torch.no_grad():
            out_r = dino_model(**dino_inputs)
    ratio_top5[r] = dict(topk_classes(out_r.logits, dino_processor, k=10))

# 출력
for label, prob_full in base_top5:
    row = f'  {label[:35]:35s}  {prob_full:6.3f}'
    for r in ratios_cls:
        p = ratio_top5[r].get(label, 0.0)
        row += f'  {p:6.3f}'
    print(row)

print('─' * 60)
print(f'  Full top-1: {base_top5[0][0]}')

In [ ]:
# 확률 변화 막대 그래프
labels_plot = [lbl[:20] for lbl, _ in base_top5]
x = np.arange(len(labels_plot))
width = 0.18

fig, ax = plt.subplots(figsize=(13, 5))
probs_full = [p for _, p in base_top5]
ax.bar(x - width*1.5, probs_full, width, label='전체 (100%)', color='steelblue')

colors = ['#e67e22', '#e74c3c', '#8e44ad']
for i, r in enumerate(ratios_cls):
    probs_r = [ratio_top5[r].get(lbl, 0.0) for lbl, _ in base_top5]
    ax.bar(x + (i - 0.5) * width, probs_r, width,
           label=f'AutoGaze {r*100:.0f}%', color=colors[i])

ax.set_xticks(x)
ax.set_xticklabels(labels_plot, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('확률')
ax.set_title('DINOv2 ImageNet Top-5 확률 — AutoGaze ratio별', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## Task 4: Semantic Segmentation — SegFormer

- 모델: `nvidia/segformer-b2-finetuned-ade-512-512` (ADE20K 150 classes)
- 백본: Mix Transformer (ViT 유사 구조), 입력 512×512
- Hook 위치: `model.segformer.encoder.patch_embeddings[0].proj`
  (첫 번째 stage patch projection 출력)
- SegFormer는 4-stage hierarchical patch embedding 사용  
  → Stage 0 (가장 고해상도): 128×128=16384 tokens
- AutoGaze 14×14 → **128×128** bilinear interpolation

In [ ]:
from transformers import AutoImageProcessor, SegformerForSemanticSegmentation

SEG_MODEL = 'nvidia/segformer-b2-finetuned-ade-512-512'
seg_processor = AutoImageProcessor.from_pretrained(SEG_MODEL)
seg_model = SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL).to(DEVICE).eval()

seg_inputs = seg_processor(images=pil_img, return_tensors='pt')
seg_inputs = {k: v.to(DEVICE) for k, v in seg_inputs.items()}
inp_h_s = seg_inputs['pixel_values'].shape[-2]
GRID_S0  = inp_h_s // 4  # SegFormer stage-0 stride=4 → 512//4=128
print(f'SegFormer-B2 loaded  |  입력 {inp_h_s}px  →  stage-0 grid: {GRID_S0}×{GRID_S0}')

In [ ]:
import torch.nn.functional as F

ADE20K_PALETTE = np.random.randint(0, 255, (150, 3), dtype=np.uint8)
np.random.seed(42)
ADE20K_PALETTE = np.random.randint(0, 255, (150, 3), dtype=np.uint8)

def seg_to_rgb(logits_batch, palette):
    seg = logits_batch.argmax(dim=1)[0].cpu().numpy()  # (H, W)
    rgb = palette[seg]                                  # (H, W, 3)
    return rgb

# SegFormer의 stage-0 patch conv 출력에 hook 적용
# patch_embeddings[0].proj는 Conv2d → 출력: (B, C, H/4, W/4)
# 일반 patch token 시퀀스가 아닌 Conv feature map 형태임에 주의

class ConvFeatureSelector:
    """SegFormer처럼 Conv2d feature map 형태의 patch embedding에 gaze mask 적용."""

    def __init__(self, selector, ag_video, grid_h, grid_w):
        mask_flat = selector.compute_gaze_mask(ag_video, target_h=grid_h, target_w=grid_w)
        # (B, grid_h*grid_w) → (B, 1, grid_h, grid_w)
        self.mask_2d = mask_flat.float().reshape(-1, 1, grid_h, grid_w)

    @contextmanager
    def apply(self, module):
        mask = self.mask_2d

        def _hook(m, inp, out):
            # out: (B, C, H, W) — resize mask to match if needed
            B, C, H, W = out.shape
            m2 = F.interpolate(mask[:B], size=(H, W), mode='nearest')
            return out * m2

        handle = module.register_forward_hook(_hook)
        try:
            yield
        finally:
            handle.remove()


from contextlib import contextmanager

seg_results = {}

# 전체 토큰
with torch.no_grad():
    out_s = seg_model(**seg_inputs)
logits_up = F.interpolate(out_s.logits, size=inp_h_s, mode='bilinear', align_corners=False)
seg_results['전체 (100%)'] = seg_to_rgb(logits_up, ADE20K_PALETTE)

# ratio별
stage0_proj = seg_model.segformer.encoder.patch_embeddings[0].proj
for ratio in [0.75, 0.5, 0.25]:
    sel = AutoGazeTokenSelector(ag_model, gazing_ratio=ratio)
    conv_sel = ConvFeatureSelector(sel, ag_video, GRID_S0, GRID_S0)
    with conv_sel.apply(stage0_proj):
        with torch.no_grad():
            out_s = seg_model(**seg_inputs)
    logits_up = F.interpolate(out_s.logits, size=inp_h_s, mode='bilinear', align_corners=False)
    seg_results[f'AutoGaze {ratio*100:.0f}%'] = seg_to_rgb(logits_up, ADE20K_PALETTE)

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
axes[0].imshow(pil_img)
axes[0].set_title('원본', fontweight='bold')
axes[0].axis('off')
for ax, (label, seg_img) in zip(axes[1:], seg_results.items()):
    ax.imshow(seg_img)
    ax.set_title(label, fontweight='bold')
    ax.axis('off')
fig.suptitle('SegFormer-B2 ADE20K — AutoGaze ratio별 세그멘테이션',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. 종합 비교: Ratio별 성능 요약

ratio가 낮을수록 더 적은 토큰 사용 → 연산량 감소  
어느 task/ratio에서 성능이 얼마나 떨어지는지 정리합니다.

In [ ]:
# Depth RMSE (전체 대비)
ref_depth = depth_results['전체 (100%)']
ratios_all = [1.0, 0.75, 0.5, 0.25]
depth_rmse = [0.0]  # ratio=1.0
for label in ['AutoGaze 75%', 'AutoGaze 50%', 'AutoGaze 25%']:
    depth_rmse.append(np.sqrt(((depth_results[label] - ref_depth)**2).mean()))

# Detection: 검출된 박스 수 정규화
n_boxes_full = n_boxes.get('100%', 1)
det_ratio_list = [
    1.0,
    n_boxes.get('75', n_boxes_full) / max(n_boxes_full, 1),
    n_boxes.get('50', n_boxes_full) / max(n_boxes_full, 1),
    n_boxes.get('25', n_boxes_full) / max(n_boxes_full, 1),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Depth RMSE
ax = axes[0]
ax.plot([r*100 for r in ratios_all], depth_rmse, 'o-', color='#2e86c1', lw=2, ms=7)
ax.set_xlabel('AutoGaze ratio (%)')
ax.set_ylabel('RMSE vs 전체 토큰')
ax.set_title('Depth Anything V2\n토큰 감소 → depth RMSE', fontweight='bold')
ax.set_xticks([r*100 for r in ratios_all])
ax.invert_xaxis()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Detection box count ratio
ax = axes[1]
ax.plot([r*100 for r in ratios_all], det_ratio_list, 's-', color='#e67e22', lw=2, ms=7)
ax.set_xlabel('AutoGaze ratio (%)')
ax.set_ylabel('검출 박스 수 (전체 대비 비율)')
ax.set_title('YOLOS 객체 탐지\n토큰 감소 → 검출 박스 수', fontweight='bold')
ax.set_xticks([r*100 for r in ratios_all])
ax.set_ylim(0, 1.2)
ax.axhline(1.0, color='gray', ls='--', lw=1, alpha=0.6)
ax.invert_xaxis()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

fig.suptitle('AutoGaze Token Ratio → Task 성능 영향', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n토큰 수별 이론적 FLOPs 감소 (attention O(N²) 기준):')
for r in ratios_all:
    n_tok = int(GRID_DA**2 * r)
    flops_ratio = r**2
    print(f'  ratio={r:.2f}  →  {n_tok}/{GRID_DA**2} tokens  →  attention FLOPs ≈ {flops_ratio*100:.0f}%')

---
## 6. 정리 및 다음 단계

### 이번에 확인한 것

1. **Forward hook만으로** 기존 pretrained decoder 코드 수정 없이 AutoGaze 토큰 선택 적용 가능
2. **Depth 추정**: ratio 0.5~0.75 수준에서도 전체 토큰 대비 품질 유지
3. **객체 탐지**: gaze 영역 내 주요 객체는 낮은 ratio에서도 검출 유지
4. **분류**: CLS 토큰이 주로 사용되므로 patch masking 영향 작음 (이미 DINO 설계 특성)

### 구현된 파일 구조

```
autogaze/models/autogaze/autogaze_cv.py
    AutoGazeTokenSelector   ← 이번 노트북의 핵심
    AutoGazeEncoder         ← 향후 새 decoder 학습용

autogaze/decoders/          ← 향후 학습 실험용 (미리 준비)
    RecognitionDecoder
    DetectionDecoder
    SegmentationDecoder
    DepthDecoder
```

### 다음 단계 제안

1. **더 긴 비디오로 테스트**: T > 1 프레임, 프레임별 gaze map 변화 확인
2. **Task-aware gaze 학습**: task loss를 AutoGaze GRPO reward로 → task에 특화된 gaze 패턴
3. **더 강한 백본**: NVILA의 SigLIP ViT (392px, 1024-dim) + AutoGaze → detection/depth
4. **Speed 측정**: 실제 latency 비교 (현재는 hook으로 계산은 그대로이므로, 진짜 sparse attention 구현 필요)
